In [ ]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd

class WellboreVisualizer:
    """
    Sebuah kelas profesional untuk memvisualisasikan arsitektur sumur bor 3D
    dengan pandangan terpotong (cutaway view) 270 derajat.
    """
    def __init__(self, fig_title="3D WELL ARCHITECTURE PLOT"):
        self.fig = go.Figure()
        self.fig.update_layout(
            title=fig_title,
            paper_bgcolor='rgba(0,0,0,0)',
            plot_bgcolor='rgba(0,0,0,0)'
        )

    @staticmethod
    def _create_mesh(r, z_start, z_end, theta_start=0, theta_end=270, points=100):
        """Membuat meshgrid koordinat untuk permukaan dinding silinder."""
        theta = np.linspace(theta_start, theta_end, points) * np.pi / 180
        z = np.linspace(z_start, z_end, points)
        theta_mesh, z_mesh = np.meshgrid(theta, z)
        x_mesh = r * np.cos(theta_mesh)
        y_mesh = r * np.sin(theta_mesh)
        return x_mesh, y_mesh, z_mesh

    @staticmethod
    def _create_cap_mesh(r_in, r_out, depth, theta_start=0, theta_end=270, points=100):
        """Membuat meshgrid koordinat untuk permukaan penutup silinder (ring)."""
        theta = np.linspace(theta_start, theta_end, points) * np.pi / 180
        r = np.linspace(r_in, r_out, points)
        r_mesh, theta_mesh = np.meshgrid(r, theta)
        x_mesh = r_mesh * np.cos(theta_mesh)
        y_mesh = r_mesh * np.sin(theta_mesh)
        z_mesh = np.full_like(x_mesh, depth)
        return x_mesh, y_mesh, z_mesh

    def add_hollow_cylinder(self, r_inner, r_outer, z_start, z_end, name, color, opacity=1.0):
        """
        Menambahkan silinder berongga 270 derajat (casing/semen) ke figur.
        Membuat dinding dalam, dinding luar, tutup atas, dan tutup bawah.
        """
        surface_points = 100
        cap_points = 271 # Resolusi tinggi untuk penutup melingkar

        colorscale = [[0, color], [1, color]]

        # 1. Dinding Luar
        X_out, Y_out, Z_out = self._create_mesh(r_outer, z_start, z_end, theta_start=0, theta_end=270, points=surface_points)
        self.fig.add_trace(go.Surface(x=X_out, y=Y_out, z=Z_out, colorscale=colorscale, showlegend=False, showscale=False, opacity=opacity, hoverinfo='name+z'))

        # 2. Dinding Dalam
        X_in, Y_in, Z_in = self._create_mesh(r_inner, z_start, z_end, theta_start=0, theta_end=270, points=surface_points)
        self.fig.add_trace(go.Surface(x=X_in, y=Y_in, z=Z_in, colorscale=colorscale, showlegend=False, showscale=False, opacity=opacity, hoverinfo='name+z'))

        # 3. Tutup Atas
        X_t, Y_t, Z_t = self._create_cap_mesh(r_inner, r_outer, z_start, theta_start=0, theta_end=270, points=cap_points)
        self.fig.add_trace(go.Surface(x=X_t, y=Y_t, z=Z_t, colorscale=colorscale, showlegend=False, showscale=False, opacity=opacity, hoverinfo='name+z'))

        # 4. Tutup Bawah
        X_b, Y_b, Z_b = self._create_cap_mesh(r_inner, r_outer, z_end, theta_start=0, theta_end=270, points=cap_points)
        self.fig.add_trace(go.Surface(x=X_b, y=Y_b, z=Z_b, colorscale=colorscale, showlegend=False, showscale=False, opacity=opacity, hoverinfo='name+z'))
        
        # Tambahkan satu trace 'kosong' ke legenda untuk mengelola seluruh grup casing
        self.fig.add_trace(go.Surface(x=[0], y=[0], z=[0], colorscale=colorscale, name=name, showlegend=True, showscale=False, opacity=0))

    def add_well_from_dataframe(self, well_df):
        """Menambahkan semua bagian casing dan semen dari Pandas DataFrame."""
        for _, row in well_df.iterrows():
            is_cement = 'Cement' in row['Type']
            
            # Pengaturan profesional untuk semen vs logam
            current_opacity = 0.5 if is_cement else 1.0
            current_color = '#A9A9A9' if is_cement else row['Color'] # Grey untuk semen secara default
            
            self.add_hollow_cylinder(
                r_inner=row['Inner Radius'],
                r_outer=row['Outer Radius'],
                z_start=row['Start Depth'],
                z_end=row['End Depth'],
                name=row['Name'],
                color=current_color,
                opacity=current_opacity
            )

    def show_plot(self):
        """Konfigurasi layout profesional dan tampilkan plot."""
        self.fig.update_layout(
            scene=dict(
                xaxis=dict(title='X', gridcolor='rgba(200,200,200,0.3)', nticks=4),
                yaxis=dict(title='Y', gridcolor='rgba(200,200,200,0.3)', nticks=4),
                zaxis=dict(
                    title='Depth (FT)',
                    autorange='reversed', # Membalik sumbu Z agar kedalaman ke bawah
                    gridcolor='rgba(200,200,200,0.3)'
                ),
                aspectmode='manual',
                aspectratio=dict(x=1, y=1, z=3) # Skala sumbu Z agar lebih representatif
            ),
            legend=dict(
                x=0.85, y=0.5,
                traceorder='normal',
                bgcolor='rgba(255,255,255,0)',
                font=dict(size=12)
            ),
            margin=dict(l=0, r=0, b=0, t=40) # Mengurangi margin untuk area plot yang lebih luas
        )
        self.fig.show()

# --- Contoh Penggunaan (Data Sampel Berbasis Spesifikasi Gambar) ---

# Definisikan data sumur dalam DataFrame Pandas
# Radius dilebih-lebihkan untuk visibilitas.
data = {
    'Type': [
        'Casing', 'Cement',  # Conductor
        'Casing', 'Cement',  # Surface
        'Casing', 'Cement',  # Intermediate
        'Casing', 'Cement',  # Production
        'Casing'              # Liner
    ],
    'Name': [
        'Conductor Casing', 'Conductor Cement',
        'Surface Casing', 'Surface Cement',
        'Intermediate Casing', 'Intermediate Cement',
        'Production Casing', 'Production Cement',
        'Liner'
    ],
    # Radius Dalam (r_in) dan Luar (r_out) diatur secara bertingkat.
    'Inner Radius': [4.5, 4.6, 3.5, 3.6, 2.5, 2.6, 1.8, 1.9, 1.0],
    'Outer Radius': [5.0, 5.0, 4.0, 4.0, 3.0, 3.0, 2.2, 2.2, 1.5],
    # Kedalaman diatur bertingkat ke bawah.
    'Start Depth': [0, 0, 0, 0, 1000, 1000, 2500, 2500, 4000],
    'End Depth': [500, 500, 1500, 1500, 3000, 3000, 4500, 4500, 6000],
    # Skema warna logam agar sesuai dengan legenda gambar
    'Color': [
        '#BF3F3F', '#A9A9A9', # Merah
        '#3FBF7F', '#A9A9A9', # Hijau
        '#BFBF3F', '#A9A9A9', # Kuning
        '#7F3FBF', '#A9A9A9', # Ungu
        '#BF3FBF'              # Pink/Magenta
    ]
}

well_df = pd.DataFrame(data)

# Inisialisasi visualizer, tambahkan data, dan tampilkan
well_plotter = WellboreVisualizer()
well_plotter.add_well_from_dataframe(well_df)
well_plotter.show_plot()